In [11]:
import pandas as pd

citation_df = pd.read_parquet(
    "../../data/processed/citation dataset/citation_dataset_cleaned.parquet"
)

print(citation_df.shape)

(17626, 4)


In [12]:
import re

def extract_case_references(text):

    if pd.isna(text):
        return []

    pattern = r'([A-Z][A-Za-z0-9.,&()\'\- ]{2,80}\s+vs\s+[A-Z][A-Za-z0-9.,&()\'\- ]{2,80})'

    matches = re.findall(pattern, str(text))

    cleaned = []

    for m in matches:

        case = m.strip()

        # Remove common trailing junk
        case = re.sub(r'\s+on\s+\d+.*$', '', case)
        case = re.sub(r'\.\.\..*$', '', case)

        # Keep only reasonable case names
        if " vs " in case and len(case) > 15:
            cleaned.append(case)

    return list(set(cleaned))

In [13]:
import random

samples = []

for i in random.sample(range(len(df)), 100):

    cases = extract_case_references(
        df.iloc[i]["legal_summary"]
    )

    samples.extend(cases)

print(samples[:50])

['Kamal Prasad vs State Of M.P. (Now Chhattisgarh)', 'Dismissing the appeal by special leave, the Court Saraswati Devi & Ors vs State Of U.P. & Ors', 'Collector, on his own motion or otherwise, of Sushila Devi vs Ramanandan Prasad & Ors', "Foreigners' Act, 1946 to leave India. He Mohd. Raza Dabstani vs State Of Bombay And Ors", 'Act, 1989. The Committee met on March 27, 1990 and The Chancellor And Anr. vs Dr Bijayananda Kar And Ors.', 'Sita Ram Bhau Patil vs Ramchandra Nago Patil (Dead) By L. Rs. & ', 'Satyender vs Saroj', 'State Of Madras vs A.R. Srinivasan', 'Parul Nahar to Soumitra Kumar Nahar vs Parul Nahar', 'Sathya Narayanan vs State Tr.Insp.Of Police', 'Lucknow Bench, which had partially accepted the appeal by Mano Dutt & Anr vs State Of U.P', 'Commissioner of Income Tax, by filing O.A. No.290 of Prabhu Dayal Khandelwal vs Chairman, U.P.S.C. & Ors', 'Prabhu Dayal Khandelwal vs Chairman, U.P.S.C. & Ors', 'Protection Officer or the service provider. Krishna Bhatacharjee vs Sarathi

In [14]:
citation_df["cited_case"].nunique()

17541

In [15]:
top = citation_df["cited_case"].value_counts()

print("Unique Cases :", citation_df["cited_case"].nunique())

print("\nTop 100 Citations:\n")

print(top.head(100))

Unique Cases : 17541

Top 100 Citations:

cited_case
Ltd vs Commissioner Of Income                                                                                                   12
State of Bihar & others vs Bihar Rajya M.S.E.S.K.K.M & others                                                                   11
Ltd vs The Commissioner Of Income                                                                                                6
Ltd vs Collector Of Central Excise                                                                                               6
Ltd vs Commissioner Of Income Tax                                                                                                5
                                                                                                                                ..
Saheb Khan vs Mohd. Yousufuddin & Ors on                                                                                         1
Isher Kaur had no right to gif

In [17]:
import re

def normalize_case_name(text):

    text = str(text)

    # remove "on 12 March 2005" type endings
    text = re.sub(r'\s+on.*$', '', text)

    # remove common suffixes
    text = re.sub(r'& Ors\.?', '', text)
    text = re.sub(r'& Anr\.?', '', text)
    text = re.sub(r'Etc\. Etc\.?', '', text)

    # normalize spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [18]:
citation_df["normalized_case"] = (
    citation_df["cited_case"]
    .apply(normalize_case_name)
)

print(
    citation_df["normalized_case"]
    .nunique()
)

17496


In [19]:
citation_df["normalized_case"] = (
    citation_df["cited_case"]
    .apply(normalize_case_name)
)

print(
    "Unique Before :",
    citation_df["cited_case"].nunique()
)

print(
    "Unique After  :",
    citation_df["normalized_case"].nunique()
)

Unique Before : 17541
Unique After  : 17496


In [20]:
citation_df["normalized_case"].value_counts().head(20)

normalized_case
Ltd vs Commissioner Of Income                                    12
State of Bihar & others vs Bihar Rajya M.S.E.S.K.K.M & others    11
Ltd vs Collector Of Central Excise                                7
Ltd vs The Commissioner Of Income                                 6
Ltd vs Union Of India                                             5
T.N. Godavarman Thirumulpad vs Union Of India                     5
Ltd vs Commissioner Of Income Tax                                 5
Bombay vs Commissioner Of Income                                  5
M.C. Mehta vs Union Of India                                      4
Court                                                             3
State Of Punjab vs Mohinder Singh                                 3
Ltd. vs Union Of India                                            3
Etc vs Union Of India                                             3
Ram Jethmalani vs Union Of India                                  3
T.N. Godavarman Thirumulpad vs U